In [1]:
import xarray as xr
from atlas.mask import mask_netcdf

In [2]:
utci = mask_netcdf(
    netcdf_path="../data/raw/ECMWF_utci_2022_mexico_anual.nc",
    shapefile_path="../data/raw/mexico_mask.gpkg"
)
utci


<xarray.Dataset> Size: 359MB
Dimensions:      (time: 8760, lon: 133, lat: 77)
Coordinates:
  * time         (time) datetime64[ns] 70kB 2022-01-01 ... 2022-12-31T23:00:00
  * lon          (lon) float64 1kB -119.0 -118.8 -118.5 ... -86.5 -86.25 -86.0
  * lat          (lat) float64 616B 33.0 32.75 32.5 32.25 ... 14.5 14.25 14.0
    spatial_ref  int64 8B 0
Data variables:
    utci         (time, lat, lon) float32 359MB nan nan nan nan ... nan nan nan
Attributes:
    CDI:                       Climate Data Interface version 1.9.8 (https://...
    Conventions:               CF-1.6
    institution:               European Centre for Medium-Range Weather Forec...
    CDO:                       Climate Data Operators version 1.9.8 (https://...
    cdo_openmp_thread_number:  8
    history:                   Mon Apr  4 07:54:18 2022: ncatted -a _FillValu...
    NCO:                       netCDF Operators version 4.7.8 (Homepage = htt...

In [3]:
UTCI_LEVELS = {
    0: (float("-inf"), -40, "Estrés por frío extremo"),
    1: (-40, -27, "Estrés por frío muy fuerte"),
    2: (-27, -13, "Estrés por frío fuerte"),
    3: (-13, 0, "Estrés por frío moderado"),
    4: (0, 9, "Estrés por frío ligero"),
    5: (9, 26, "Sin estrés térmico"),
    6: (26, 32, "Estrés por calor moderado"),
    7: (32, 38, "Estrés por calor fuerte"),
    8: (38, 46, "Estrés por calor muy fuerte"),
    9: (46, float("inf"), "Estrés por calor extremo"),
}


In [4]:
categories = xr.full_like(utci, float("nan"))

for category, (lim_inf, lim_sup, _) in UTCI_LEVELS.items():
    mask = (utci > lim_inf) & (utci <= lim_sup)

    categories = xr.where(mask,category,categories,)

categories = categories.astype("float32")
categories

<xarray.Dataset> Size: 359MB
Dimensions:      (time: 8760, lat: 77, lon: 133)
Coordinates:
  * time         (time) datetime64[ns] 70kB 2022-01-01 ... 2022-12-31T23:00:00
  * lat          (lat) float64 616B 33.0 32.75 32.5 32.25 ... 14.5 14.25 14.0
  * lon          (lon) float64 1kB -119.0 -118.8 -118.5 ... -86.5 -86.25 -86.0
    spatial_ref  int64 8B 0
Data variables:
    utci         (time, lat, lon) float32 359MB nan nan nan nan ... nan nan nan

In [5]:
valid_count = categories.notnull().sum("time")

In [6]:
counts = xr.concat(
    [
        (categories == category).sum("time")
        for category in UTCI_LEVELS
    ],
    dim="category",
)


In [7]:

counts = counts.assign_coords(category=list(UTCI_LEVELS.keys()))
UTCI_LEVELS.keys


<function dict.keys()>

In [8]:
stress_percentage = (counts / valid_count * 100)

In [9]:
labels = [info[2] for info in UTCI_LEVELS.values()]

stress_percentage = stress_percentage.assign_coords(
    category_label=("category", labels)
)
stress_percentage

<xarray.Dataset> Size: 822kB
Dimensions:         (lon: 133, lat: 77, category: 10)
Coordinates:
  * lon             (lon) float64 1kB -119.0 -118.8 -118.5 ... -86.25 -86.0
  * lat             (lat) float64 616B 33.0 32.75 32.5 32.25 ... 14.5 14.25 14.0
  * category        (category) int64 80B 0 1 2 3 4 5 6 7 8 9
    category_label  (category) <U27 1kB 'Estrés por frío extremo' ... 'Estrés...
    spatial_ref     int64 8B 0
Data variables:
    utci            (category, lat, lon) float64 819kB nan nan nan ... nan nan

In [10]:
pct_table = (
    stress_percentage
    .to_dataframe()
    .reset_index()
    .dropna(subset=["utci"])
)

pct_table = pct_table[["lat", "lon", "category_label", "utci"]]

display(
    pct_table.sort_values(
        ["lat", "lon", "utci"],
        ascending=[True, True, False]
    ).head(30)
)

,lat,lon,category_label,utci
83125,14.75,-92.25,Sin estrés térmico,34.988584
83127,14.75,-92.25,Estrés por calor fuerte,34.006849
83126,14.75,-92.25,Estrés por calor moderado,26.986301
83128,14.75,-92.25,Estrés por calor muy fuerte,4.018265
83120,14.75,-92.25,Estrés por frío extremo,0.000000
83121,14.75,-92.25,Estrés por frío muy fuerte,0.000000
83122,14.75,-92.25,Estrés por frío fuerte,0.000000
83123,14.75,-92.25,Estrés por frío moderado,0.000000
83124,14.75,-92.25,Estrés por frío ligero,0.000000
83129,14.75,-92.25,Estrés por calor extremo,0.000000


In [11]:
# stress_percentage.to_netcdf("../data/processed/porcentaje_utci.nc")

In [12]:
f = ("../data/processed/porcentaje_utci.nc")
utci = xr.open_dataset(f)

In [13]:
utci

<xarray.Dataset> Size: 822kB
Dimensions:         (category: 10, lat: 77, lon: 133)
Coordinates:
  * category        (category) int64 80B 0 1 2 3 4 5 6 7 8 9
    category_label  (category) <U27 1kB ...
  * lat             (lat) float64 616B 33.0 32.75 32.5 32.25 ... 14.5 14.25 14.0
  * lon             (lon) float64 1kB -119.0 -118.8 -118.5 ... -86.25 -86.0
    spatial_ref     int64 8B ...
Data variables:
    utci            (category, lat, lon) float64 819kB ...

In [14]:
utci.category_label.values

array(['Estrés por frío extremo', 'Estrés por frío muy fuerte',
       'Estrés por frío fuerte', 'Estrés por frío moderado',
       'Estrés por frío ligero', 'Sin estrés térmico',
       'Estrés por calor moderado', 'Estrés por calor fuerte',
       'Estrés por calor muy fuerte', 'Estrés por calor extremo'],
      dtype='<U27')

In [15]:
utci["utci"].sel(
    lat=16.75,
    lon=-96.75,
    method="nearest"
).values

array([ 0.        ,  0.        ,  0.        ,  0.        ,  9.6803653 ,
       66.41552511, 22.043379  ,  1.86073059,  0.        ,  0.        ])

In [16]:
oax= utci["utci"].sel(
    lat=17.07,
    lon=-96.72,
    method="nearest"
)

for label, percentage in zip(
    oax.category_label.values,
    oax.values
):
    print(f"{label}: {percentage:.2f}%")

Estrés por frío extremo: 0.00%
Estrés por frío muy fuerte: 0.00%
Estrés por frío fuerte: 0.00%
Estrés por frío moderado: 0.06%
Estrés por frío ligero: 9.62%
Sin estrés térmico: 67.60%
Estrés por calor moderado: 20.62%
Estrés por calor fuerte: 2.10%
Estrés por calor muy fuerte: 0.00%
Estrés por calor extremo: 0.00%
